In [1]:
import re, sys, os
from pathlib import Path
import torch
import numpy as np
repo_start = f'../'
sys.path.append(repo_start)

from modules.utils.imports import *
from modules.binn_eql.model_wrapper_2d import model_wrapper
from modules.binn_eql.build_binn_eql_net import BINN
from modules.loaders.format_data import format_data_general

In [10]:
def extract_final_loss(filepath):
    """
    Reads the given file and extracts the validation loss (as a float)
    from the line containing the word 'Elapsed'.
    """
    # Regular expression to capture the validation loss
    # This regex looks for "Val loss = " followed by a floating point number (possibly in scientific notation)
    val_loss_pattern = re.compile(r"Val loss = ([+-]?\d+(?:\.\d+)?(?:e[+-]?\d+)?)")
    
    with open(filepath, 'r') as file:
        for line in file:
            # Check if the line contains "Elapsed"
            if "Elapsed" in line:
                # Search the line for the pattern
                match = val_loss_pattern.search(line)
                if match:
                    try:
                        return float(match.group(1))
                    except ValueError:
                        print(f"Conversion error in file {filepath} for line: {line.strip()}")
                        return None
    # If no matching line was found
    return None

def extract_final_loss_fine_tuned(filepath):
    """
    Reads the given file and extracts the validation loss (as a float)
    from the line containing the word 'Elapsed'.
    """
    val_losses = []
    # Regular expression to capture the validation loss
    # This regex looks for "Val loss = " followed by a floating point number (possibly in scientific notation)
    val_loss_pattern = re.compile(r"Val loss = ([+-]?\d+(?:\.\d+)?(?:e[+-]?\d+)?)")
    
    with open(filepath, 'r') as file:
        for line in file:
            # Check if the line contains "Elapsed"
            if "Elapsed" in line:
                # Search the line for the pattern
                match = val_loss_pattern.search(line)
                if match:
                    try:
                        val_losses.append(float(match.group(1)))
                    except ValueError:
                        print(f"Conversion error in file {filepath} for line: {line.strip()}")
                        return None
        return val_losses[1] if len(val_losses) >= 2 else None
    # If no matching line was found
    return None


In [11]:
def process_directory(parent_dir, fine_tuned=False):
    """
    For each subdirectory under parent_dir, finds slurm files with the pattern "slurm-####.out",
    selects the one with the lower number, extracts the validation loss, and prints the subdirectory path
    alongside the extracted loss.
    """
    results = {}
    # Use pathlib to iterate over subdirectories
    for subdir in Path(parent_dir).iterdir():
        if subdir.is_dir():
            # Find files matching pattern "slurm-*.out"
            slurm_files = list(subdir.glob("slurm-*.out"))
            if not slurm_files:
                # print(f"No slurm files found in {subdir}")
                results[str(subdir)] = None
                continue
            
            # Extract numeric part from filename (assumes naming: slurm-####.out)
            def extract_number(file_path):
                match = re.search(r"slurm-(\d+)\.out", file_path.name)
                return int(match.group(1)) if match else float('inf')
            
            # Sort files by the extracted number
            slurm_files.sort(key=extract_number)
            # Take the first file (lowest number)
            first_file = slurm_files[0]
            if fine_tuned:
                val_loss = extract_final_loss_fine_tuned(first_file)
            else:
                val_loss = extract_final_loss(first_file)
            if val_loss is not None:
                results[str(subdir)] = val_loss
            else:
                # print(f"Could not extract validation loss from file {first_file} in {subdir}")
                results[str(subdir)] = None
    
    return results


In [12]:
# Load training data for removing terms
training_data_path = '/work/users/s/m/smyersn/elston/projects/kinetics_binns/data/2d/random_data_non_negative.npz'
training_data = format_data_general(2, 2, file=training_data_path)

u_triangle_mesh, v_triangle_mesh = lltriangle(training_data[:, -2:], 
                                                training_data[:, -1:])
# Create 1d arrays from meshes
u_triangle, v_triangle = np.ravel(u_triangle_mesh), np.ravel(v_triangle_mesh)

# Create separate variables for arrays containing and not containing nans
uv_nans = np.stack((u_triangle, v_triangle), axis=1)
mask = ~np.isnan(uv_nans).any(axis=1)
uv = torch.from_numpy(uv_nans[mask]).to('cpu')

In [19]:
parent_directory = '/work/users/s/m/smyersn/elston/projects/kinetics_binns/development/binn_eql_net/runs/debugging/26_pruning_every_10k'
losses = process_directory(parent_directory, fine_tuned=False)

# Sort the results by validation loss (lowest first)
sorted_losses = sorted(losses.items(), key=lambda item: (item[1] is None, item[1]))

# Print the sorted results
for dir_name, val_loss in sorted_losses:
    if val_loss:
        print(f"\nDirectory: {dir_name.split('/')[-1]}, Validation Loss: {val_loss}")
        
        # Load model
        binn = BINN(
            dimensions=2,
            species=2, 
            train_data=torch.tensor(training_data), 
            diff_coeffs=[0.01, 1],
            duplicates=1)

        binn.to('cpu')

        parameters = binn.parameters()

        opt = torch.optim.Adam(parameters, lr=0.001)

        model = model_wrapper(
            model=binn,
            optimizer=opt,
            loss=binn.loss,
            dir_name=dir_name,
            save_name=f'{dir_name}/binn')
                
        model.load(f"{dir_name}/binn_best_val_model", device='cpu')
        
        # Print equation
        fn = f'{dir_name}/equation.txt'
        model.model.remove_insignificant_terms(uv)
        model.model.fix_cheating_hill_functions(uv)
        for term in model.model.generate_equation():
            print(f'{term}')


Directory: binn_eql_gls_1_pde_1_repeat_14, Validation Loss: 0.078085
-0.075 * u * u
-2.503 * u^1.000 / (1 + 0.338 * u^1.000)
2.301 * u * (1 / 1.164 - v^0.896 / (1 + 1.164 * v^0.896))

Directory: binn_eql_gls_1_pde_1_repeat_38, Validation Loss: 0.10003
0.989 * u * v
-2.730 * v * u^0.694 / (1 + 0.427 * u^0.694)

Directory: binn_eql_gls_1_pde_1_repeat_27, Validation Loss: 0.10853
0.040 * u * u
-1.801 * u * (1 / 4.780 - v^1.184 / (1 + 4.780 * v^1.184))

Directory: binn_eql_gls_1_pde_1_repeat_45, Validation Loss: 0.11016
1.061 * u * v
-0.093 * (1 / 0.053 - v^1.239 / (1 + 0.053 * v^1.239))

Directory: binn_eql_gls_1_pde_1_repeat_41, Validation Loss: 0.12753
1.158 * u
-3.498 * u * v

Directory: binn_eql_gls_1_pde_1_repeat_35, Validation Loss: 0.12893
1.136 * u
-3.731 * u * v

Directory: binn_eql_gls_1_pde_1_repeat_7, Validation Loss: 0.13006
1.102 * u
-3.623 * u * v

Directory: binn_eql_gls_1_pde_1_repeat_31, Validation Loss: 0.1314
0.012 * u * u

Directory: binn_eql_gls_1_pde_1_repeat_30, V

In [ ]:
the_dir = '/work/users/s/m/smyersn/elston/projects/kinetics_binns/development/binn_eql_net/runs/debugging/20_l05_sweep/binn_eql_gls_1_pde_1_l05_0.01_repeat_25'

binn = BINN(
    dimensions=2,
    species=2, 
    train_data=torch.tensor(training_data), 
    diff_coeffs=[0.01, 1],
    duplicates=1)

binn.to('cpu')

parameters = binn.parameters()

opt = torch.optim.Adam(parameters, lr=0.001)

model = model_wrapper(
    model=binn,
    optimizer=opt,
    loss=binn.loss,
    dir_name=dir_name,
    save_name=f'{dir_name}/binn')

model.load(f"{the_dir}/binn_best_val_model", device='cpu')

print(f'Original equation:')
for term in model.model.generate_equation():
    print(f'{term}')

print(f'\nOriginal equation manually corrected:')
model.model.remove_insignificant_terms(uv)
model.model.fix_cheating_hill_functions(uv)
for term in model.model.generate_equation():
    print(f'{term}')

model.load(f"{the_dir}/binn_best_val_fine_tuned_model", device='cpu')

print(f'\nFine-tuned equation:')
for term in model.model.generate_equation():
    print(f'{term}')
    
print(f'\nFine-tuned equation manually corrected:')
model.model.remove_insignificant_terms(uv)
model.model.fix_cheating_hill_functions(uv)
for term in model.model.generate_equation():
    print(f'{term}')

Original equation:
-0.001 * u
0.000 * v
0.022 * u * u
0.000 * v * v
0.982 * u * v
-0.005 * u^0.367 / (1 + 0.000 * u^0.367)
-0.000 * v^4.992 / (1 + 9.901 * v^4.992)
-0.000 * v * u^0.485 / (1 + 0.979 * u^0.485)
0.004 * u * v^1.666 / (1 + 0.000 * v^1.666)
-0.052 * (1 / 0.007 - u^3.037 / (1 + 0.007 * u^3.037))
0.496 * (1 / 0.069 - v^2.224 / (1 + 0.069 * v^2.224))
0.003 * v * (1 / 0.007 - u^0.211 / (1 + 0.007 * u^0.211))
-0.051 * u * (1 / 0.054 - v^1.613 / (1 + 0.054 * v^1.613))

Original equation manually corrected:
-0.952 * u
0.982 * u * v
-0.052 * (1 / 0.007 - u^3.037 / (1 + 0.007 * u^3.037))
0.496 * (1 / 0.069 - v^2.224 / (1 + 0.069 * v^2.224))

Fine-tuned equation:
-0.937 * u
0.023 * u * u
1.052 * u * v
-0.040 * (1 / 0.006 - u^3.189 / (1 + 0.006 * u^3.189))
0.218 * (1 / 0.032 - v^2.866 / (1 + 0.032 * v^2.866))

Fine-tuned equation manually corrected:
-0.937 * u
1.052 * u * v
-0.040 * (1 / 0.006 - u^3.189 / (1 + 0.006 * u^3.189))
0.218 * (1 / 0.032 - v^2.866 / (1 + 0.032 * v^2.866))


In [8]:
from modules.symbolic_net.custom_norm import custom_norm_old, custom_norm
torch.set_printoptions(precision=10)

test = torch.tensor([0.00431, 0.05, 0.047, 3, 0, 0.58, 34, 0.0000034, 0.05])
print(custom_norm_old(test))
print(custom_norm(test))

tensor(8.8753271103)
tensor(8.8753271103)


In [17]:
coeffs = model.model.reaction.eql_layer.fc.weight
coeff_loss = (coeffs - coeffs.clamp(-10, 10))**2

D_loss = torch.tensor(0.0, device=coeffs.device)

total_param_loss = torch.mean(coeff_loss + D_loss)


In [18]:
print(coeff_loss)
print(D_loss)

tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]],
       grad_fn=<PowBackward0>)
tensor(0.)
